Here chi=20 - no damping, fixing the number of Gilt R-matrix iterations. Newton method converges, starting from T=T_c, doing 4 RG iterations, then Newton. Using 5 eigenvalues here

In [5]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [2]:
gilt_eps = 2e-5 # from the paper for this chi
chi = 20
cg_eps = 1e-10
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 0,
	"rotate" => true,
)
Jratio = 1.0

relT=1.0
#do 4 steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, 4, gilt_pars)["A"];
#NB traj consists of PyObjects

traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
traj[5], accepted_elements, _ = fix_discrete_gauge(traj[5]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

In [ ]:
Newton iterations (I interrupted the code after a few iterations, but in previous runs I saw it converge)

In [6]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = traj[4]
for i in 1:20
    A[i], accepted_elements[i] = fix_discrete_gauge(A[i]; tol = 1e-7);
    RA = gilt(A[i], accepted_elements[i], gilt_pars);
    println("i=",i) 
    println("||R(A[i])-A[i]||= ", embedded_distance(RA, A[i]))
    flush(stdout)
    deltaA[i] = newton_correction_with_iterations_fixed(A[i], 5, accepted_elements[i], gilt_pars; gmres = true, 
        linsolvetol = 1e-8);
    println("||deltaA[i]||= ", norm(deltaA[i]))
    A[i+1] = A[i] + deltaA[i]
end

i=1
||R(A[i])-A[i]||= 0.05785177411987096
Dict{Any, Any}((1, "N") => 28, (1, "W") => 22, (1, "S") => 26, (1, "E") => 22, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


[ Info: GMRES linsolve in iter 1; step 1: normres = 8.698702663243e-02
[ Info: GMRES linsolve in iter 1; step 2: normres = 4.190304825673e-02
[ Info: GMRES linsolve in iter 1; step 3: normres = 4.185674802133e-02
[ Info: GMRES linsolve in iter 1; step 4: normres = 2.201848499021e-02
[ Info: GMRES linsolve in iter 1; step 5: normres = 1.265721107202e-02
[ Info: GMRES linsolve in iter 1; finished at step 5: normres = 1.265721107202e-02
[ Info: GMRES linsolve in iter 2; step 1: normres = 7.643552853991e-03
[ Info: GMRES linsolve in iter 2; step 2: normres = 5.338973004661e-03
[ Info: GMRES linsolve in iter 2; step 3: normres = 5.178476655025e-03
[ Info: GMRES linsolve in iter 2; step 4: normres = 2.674726321048e-03
[ Info: GMRES linsolve in iter 2; step 5: normres = 3.347076641000e-04
[ Info: GMRES linsolve in iter 2; finished at step 5: normres = 3.347076641000e-04
[ Info: GMRES linsolve in iter 3; step 1: normres = 1.430962245239e-04
[ Info: GMRES linsolve in iter 3; step 2: normres = 8

||deltaA[i]||= 0.14203719138085996
i=2
||R(A[i])-A[i]||= 0.028310162070464686
Dict{Any, Any}((1, "N") => 77, (1, "W") => 49, (1, "S") => 58, (1, "E") => 48, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


[ Info: GMRES linsolve in iter 1; step 1: normres = 1.587484952335e-01
[ Info: GMRES linsolve in iter 1; step 2: normres = 7.196028343665e-02
[ Info: GMRES linsolve in iter 1; step 3: normres = 5.591940343613e-02
[ Info: GMRES linsolve in iter 1; step 4: normres = 5.381407410694e-02
[ Info: GMRES linsolve in iter 1; step 5: normres = 3.622057991793e-02
[ Info: GMRES linsolve in iter 1; finished at step 5: normres = 3.622057991793e-02
[ Info: GMRES linsolve in iter 2; step 1: normres = 3.121756306987e-02
[ Info: GMRES linsolve in iter 2; step 2: normres = 2.701806759117e-02
[ Info: GMRES linsolve in iter 2; step 3: normres = 1.683696124102e-02
[ Info: GMRES linsolve in iter 2; step 4: normres = 1.073412304387e-02
[ Info: GMRES linsolve in iter 2; step 5: normres = 1.054807083744e-02
[ Info: GMRES linsolve in iter 2; finished at step 5: normres = 1.054807083744e-02
[ Info: GMRES linsolve in iter 3; step 1: normres = 1.036281115562e-02
[ Info: GMRES linsolve in iter 3; step 2: normres = 7

||deltaA[i]||= 0.03730779711549154
i=3
||R(A[i])-A[i]||= 0.0053028212059820405
Dict{Any, Any}((1, "N") => 76, (1, "W") => 55, (1, "S") => 48, (1, "E") => 54, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


...

||deltaA[i]||= 3.8116883263753244e-8
i=15
||R(A[i])-A[i]||= 9.350749381482492e-9
Dict{Any, Any}((1, "N") => 72, (1, "W") => 49, (1, "S") => 45, (1, "E") => 49, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


[ Info: GMRES linsolve in iter 1; step 1: normres = 2.703356504993e-01
[ Info: GMRES linsolve in iter 1; step 2: normres = 9.680450701091e-02
[ Info: GMRES linsolve in iter 1; step 3: normres = 3.528535045434e-02
[ Info: GMRES linsolve in iter 1; step 4: normres = 2.572531605808e-02
[ Info: GMRES linsolve in iter 1; step 5: normres = 2.406748039627e-02
[ Info: GMRES linsolve in iter 1; finished at step 5: normres = 2.406748039627e-02
[ Info: GMRES linsolve in iter 2; step 1: normres = 2.403354430106e-02
[ Info: GMRES linsolve in iter 2; step 2: normres = 2.029613167856e-02
[ Info: GMRES linsolve in iter 2; step 3: normres = 1.063825967203e-02
[ Info: GMRES linsolve in iter 2; step 4: normres = 5.070609347748e-03
[ Info: GMRES linsolve in iter 2; step 5: normres = 2.168263737150e-03
[ Info: GMRES linsolve in iter 2; finished at step 5: normres = 2.168263737150e-03
[ Info: GMRES linsolve in iter 3; step 1: normres = 8.148085761744e-04
[ Info: GMRES linsolve in iter 3; step 2: normres = 5

||deltaA[i]||= 2.1062302836125546e-8
i=16
||R(A[i])-A[i]||= 5.332561196997143e-9
Dict{Any, Any}((1, "N") => 72, (1, "W") => 49, (1, "S") => 45, (1, "E") => 49, (2, "N") => 1, (2, "W") => 1, (2, "S") => 1, (2, "E") => 1)


[ Info: GMRES linsolve in iter 1; step 1: normres = 1.167994222252e-01
[ Info: GMRES linsolve in iter 1; step 2: normres = 4.182421773698e-02
[ Info: GMRES linsolve in iter 1; step 3: normres = 3.888621682035e-02
[ Info: GMRES linsolve in iter 1; step 4: normres = 2.815543242142e-02
[ Info: GMRES linsolve in iter 1; step 5: normres = 1.641502712736e-02
[ Info: GMRES linsolve in iter 1; finished at step 5: normres = 1.641502712736e-02
[ Info: GMRES linsolve in iter 2; step 1: normres = 1.376783278340e-02
[ Info: GMRES linsolve in iter 2; step 2: normres = 1.352280888340e-02
[ Info: GMRES linsolve in iter 2; step 3: normres = 1.332241728344e-02
[ Info: GMRES linsolve in iter 2; step 4: normres = 1.006672063984e-02
[ Info: GMRES linsolve in iter 2; step 5: normres = 4.997391771351e-03
[ Info: GMRES linsolve in iter 2; finished at step 5: normres = 4.997391771351e-03
[ Info: GMRES linsolve in iter 3; step 1: normres = 2.852363458247e-03
[ Info: GMRES linsolve in iter 3; step 2: normres = 2

LoadError: InterruptException: